In [1]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: PyPortfolioOpt
    Found existing installation: pyportfolioopt 1.5.6
    Uninstalling pyportfolioopt-1.5.6:
      Successfully uninstalled pyportfolioopt-1.5.6


In [2]:
!pip install scikit-learn==1.2.2
import sklearn
import pypfopt
print(sklearn.__version__)
print(pypfopt.__version__)

1.2.2
1.5.5


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [4]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [5]:
def modify_date(date):
    return date[:10]

In [6]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [7]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [8]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [9]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [10]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [11]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

In [12]:
# Preselection parameters set up
method = "K_means_multiple"          # or avg or anyother method
stock_type = "S&P500"
end_swan = "Swan10D"                 #equals the last swan before the swan we are going to predict
start_swan="Swan6"
start = dt(2023,8,28)
end = dt(2024,12,2)
selected_stock_number = 50     #number of stock preselected
start_swan_date = dt(2024,12,2)
end_swan_date = dt(2024,12,5)
w1 = 0.1
w2 = 0.1
MA_days = 30
N = 17 # This parameter is set to help classification method like K-means

# Step 1 - Preselection

In [13]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

In [14]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [15]:
portfolio_beta= portfolio_beta.loc[:,"Swan1":end_swan]
portfolio_beta

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.935350
GM,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,1.734848
KMI,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,NaN
GOOG,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,1.184058
GOOGL,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,1.201377
...,...,...,...,...,...,...,...,...,...,...
META,NaN,NaN,NaN,NaN,NaN,NaN,1.142586,1.183531,1.011852,2.289337
PYPL,NaN,NaN,NaN,NaN,NaN,NaN,1.440542,0.580067,0.989242,2.093559
CRM,NaN,NaN,NaN,NaN,0.955665,1.505787,0.782578,1.189409,0.860818,1.626909


In [16]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [17]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Swan1,1.0,-0.003532,0.112309,0.050640,0.001797,0.102327,0.079934,0.077152,0.038135,-0.097569
Swan2,NaN,1.000000,0.548025,0.335659,0.410765,0.390310,0.033350,0.307687,0.037435,0.484681
Swan3,NaN,NaN,1.000000,0.158060,0.260501,0.440204,0.249559,0.228378,0.106342,0.347335
Swan4,NaN,NaN,NaN,1.000000,-0.022423,0.156316,0.155139,0.233628,0.093825,0.303871
Swan5,NaN,NaN,NaN,NaN,1.000000,0.370946,0.254273,0.219081,0.566815,0.156276
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.226079,0.447314,0.362923,0.247842
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.138652,0.188918,-0.065778
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.207563,0.123789
Swan9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,-0.040622
Swan10D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [18]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [19]:
portfolio_beta

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.935350
GM,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,1.734848
KMI,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,NaN
GOOG,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,1.184058
GOOGL,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,1.201377
...,...,...,...,...,...,...,...,...,...,...
META,NaN,NaN,NaN,NaN,NaN,NaN,1.142586,1.183531,1.011852,2.289337
PYPL,NaN,NaN,NaN,NaN,NaN,NaN,1.440542,0.580067,0.989242,2.093559
CRM,NaN,NaN,NaN,NaN,0.955665,1.505787,0.782578,1.189409,0.860818,1.626909


In [20]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1,n_init=10)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group


# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 500 trials with davies bouldin index
    for random in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=random,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[random]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,5):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <13:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    df.to_csv('stock_frequency.csv')

    for p in range(0,min(50,len(sorted_count))):
      preselected_stock.append(sorted_count[p][0])
    preselected_stock.sort()
    print(preselected_stock)

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    preselected_stock = selected_beta
    preselected_stock.to_csv('preselected_stock.csv')


<ipython-input-20-c61a3e96e89f>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-20-c61a3e96e89f>:74: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
<ipython-input-20-c61a3e96e89f>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=se

['LLY', 'MRK', 'AMGN', 'PEP', 'KO', 'SO', 'SBUX', 'EXC', 'NEE', 'DUK', 'MDLZ', 'MCD', 'AMGN', 'MRK', 'BMY', 'LLY', 'WMT', 'SBUX', 'MCD', 'AMT', 'PEP', 'EXC', 'MDLZ', 'KO', 'SO', 'VZ', 'DUK', 'NEE', 'PG', 'LLY', 'MRK', 'SO', 'DUK', 'MDLZ', 'NEE', 'PEP', 'EXC', 'MCD', 'KO', 'SBUX', 'LLY', 'MRK', 'VZ', 'SBUX', 'SO', 'DUK', 'AMT', 'PG', 'PEP', 'MDLZ', 'EXC', 'MCD', 'KO', 'NEE', 'AMGN', 'LLY', 'MRK', 'BMY', 'SO', 'VZ', 'AMT', 'DUK', 'PG', 'MDLZ', 'PEP', 'EXC', 'MCD', 'KO', 'NEE', 'SBUX']
[('DUK', 5), ('EXC', 5), ('KO', 5), ('LLY', 5), ('MCD', 5), ('MDLZ', 5), ('MRK', 5), ('NEE', 5), ('PEP', 5), ('SBUX', 5), ('SO', 5), ('AMGN', 3), ('AMT', 3), ('PG', 3), ('VZ', 3), ('BMY', 2), ('WMT', 1)]
['AMGN', 'AMT', 'BMY', 'DUK', 'EXC', 'KO', 'LLY', 'MCD', 'MDLZ', 'MRK', 'NEE', 'PEP', 'PG', 'SBUX', 'SO', 'VZ', 'WMT']


<ipython-input-20-c61a3e96e89f>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-20-c61a3e96e89f>:74: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
<ipython-input-20-c61a3e96e89f>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=se

In [21]:
res

array([1.22979668, 0.64300172, 1.58224304, 0.42536894, 0.69962112,
       0.6922777 , 1.41624301, 0.95306274, 1.42484779, 1.70334275,
       0.31987677, 1.18055573, 0.77457893, 0.91468641, 1.26522026,
       0.77249889, 1.02418496])

In [22]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17 entries, AMGN to WMT
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Swan6    17 non-null     float64
 1   Swan7    17 non-null     float64
 2   Swan8    17 non-null     float64
 3   Swan9    17 non-null     float64
 4   Swan10D  17 non-null     float64
 5   Group    17 non-null     int32  
 6   avg      17 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 1020.0+ bytes


In [23]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

In [24]:
portfolio_beta = pd.read_csv("HoltestimateS11_SP500.csv",index_col=0)
if 'Ticker' in preselected_stock.columns:
    preselected_stock.set_index('Ticker', inplace=True)
preselected_stock['avg'] = preselected_stock.index.map(portfolio_beta['Forecasted Swan11'])

In [25]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock

,Swan6,Swan7,Swan8,Swan9,Swan10D,Group,avg,PR
Ticker,,,,,,,,
AMGN,0.109292,0.843475,0.659977,0.526728,-0.417885,10,-1.208625,2.208625
AMT,0.050751,0.583639,-0.336659,0.882155,1.271482,3,1.312267,-0.312267
BMY,-0.398965,1.296441,1.085742,0.931988,-0.566874,10,-1.841028,2.841028
DUK,-0.322793,0.233701,-0.356250,1.165454,0.534699,3,1.044984,-0.044984
EXC,0.263389,0.605246,-0.123438,1.254518,0.353268,3,0.718945,0.281055
KO,0.325968,0.614677,-0.049526,1.161009,0.181359,3,0.523833,0.476167
LLY,0.280671,0.780342,0.061466,0.490481,-0.761718,10,-0.542674,1.542674
MCD,0.130072,0.483323,-0.202955,1.142392,0.376953,3,0.731855,0.268145
MDLZ,0.344205,0.089729,0.386481,0.928976,0.451681,3,0.756474,0.243526


# Step 2 - Download data and adjuste price based on Swan Beta

In [26]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [27]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['AMGN' 'AMT' 'BMY' 'DUK' 'EXC' 'KO' 'LLY' 'MCD' 'MDLZ' 'MRK' 'NEE' 'PEP'
 'PG' 'SBUX' 'SO' 'VZ' 'WMT']


In [28]:
preselected_stock.shape

(17, 8)

In [29]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]


[*********************100%%**********************]  17 of 17 completed


In [30]:
yahooDataFrame

Ticker,AMGN,AMT,BMY,DUK,EXC,KO,LLY,MCD,MDLZ,MRK,NEE,PEP,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,
2023-08-28,247.082748,170.773407,58.083759,85.499184,38.280807,57.842083,549.313293,273.846100,69.377327,105.397667,65.110748,173.589935,149.103485,94.056396,65.057205,30.796688,52.201977
2023-08-29,250.058746,173.965149,59.223946,85.755630,38.518578,57.784779,549.293396,273.759247,69.309441,106.423294,65.070267,174.244797,149.190735,96.101517,65.477356,31.843130,52.639404
2023-08-30,248.363678,174.712769,58.781067,85.432701,38.442490,57.756123,543.432556,274.386688,69.396706,106.636162,64.665443,174.389252,149.355576,96.188744,65.525108,31.797235,53.017635
2023-08-31,246.880524,173.792633,58.093182,84.340462,38.157169,57.144844,549.590942,272.862976,69.096130,105.446053,64.385933,172.551437,149.646454,94.444099,64.675255,32.109329,53.481377
2023-09-01,247.236847,173.284637,58.441833,83.514137,38.033535,56.648182,552.476746,272.659180,67.573799,106.278152,64.453400,170.029892,149.811295,94.986885,64.188255,31.999180,53.139324
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-22,294.529999,204.309998,58.869999,114.699997,38.709999,63.438839,748.010010,288.544250,64.400002,99.180000,76.000000,162.000000,176.279999,102.500000,87.599998,43.150002,90.440002
2024-11-25,294.000000,207.550003,58.779999,115.000000,39.130001,63.895374,755.000000,294.418915,64.940002,101.160004,76.639999,163.050003,177.389999,101.839996,88.419998,43.980000,89.500000
2024-11-26,280.010010,208.000000,58.740002,117.040001,39.490002,64.064102,789.320007,294.558075,64.879997,101.620003,77.489998,162.160004,179.309998,100.680000,89.330002,44.369999,91.309998


In [31]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [32]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [33]:
market_drop.to_csv("market_drop.csv")

In [34]:
market_drop

,0
0,0.0000
1,0.0028
2,0.0055
3,0.0083
4,0.0110
...,...
313,0.0000
314,0.0000
315,0.0000
316,0.0000


In [35]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [36]:
yahooDataFrame


Ticker,AMGN,AMT,BMY,DUK,EXC,KO,LLY,MCD,MDLZ,MRK,NEE,PEP,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,
2023-08-28,247.082748,170.773407,58.083759,85.499184,38.280807,57.842083,549.313293,273.846100,69.377327,105.397667,65.110748,173.589935,149.103485,94.056396,65.057205,30.796688,52.201977
2023-08-29,250.904983,173.325941,59.529238,85.504713,38.441038,57.700024,550.128041,273.198261,69.162634,106.628273,64.904588,174.145874,148.868629,95.858388,65.314908,31.757565,52.614134
2023-08-30,250.014661,173.451785,59.376264,84.941684,38.290480,57.589722,545.054544,273.282226,69.107973,107.039603,64.342028,174.194778,148.722169,95.710735,65.205781,31.629404,52.967640
2023-08-31,249.357128,171.899716,58.980876,83.608946,37.929476,56.896389,552.066409,271.205498,68.662294,106.048088,63.899979,172.261052,148.688724,93.735825,64.199611,31.853570,53.405269
2023-09-01,250.523831,170.783285,59.625357,82.554157,37.732751,56.321766,555.774710,270.464162,67.011503,107.082327,63.808690,169.650668,148.540615,94.042814,63.562631,31.661385,53.039103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-22,294.529999,204.309998,58.869999,114.699997,38.709999,63.438839,748.010010,288.544250,64.400002,99.180000,76.000000,162.000000,176.279999,102.500000,87.599998,43.150002,90.440002
2024-11-25,294.000000,207.550003,58.779999,115.000000,39.130001,63.895374,755.000000,294.418915,64.940002,101.160004,76.639999,163.050003,177.389999,101.839996,88.419998,43.980000,89.500000
2024-11-26,280.010010,208.000000,58.740002,117.040001,39.490002,64.064102,789.320007,294.558075,64.879997,101.620003,77.489998,162.160004,179.309998,100.680000,89.330002,44.369999,91.309998


In [37]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['AMGN', 'AMT', 'BMY', 'DUK', 'EXC', 'KO', 'LLY', 'MCD', 'MDLZ',
       'MRK', 'NEE', 'PEP', 'PG', 'SBUX', 'SO', 'VZ', 'WMT'], dtype=object)

# Step3 - MVO

In [38]:
#Set the paramter for MVO process
Portfolio_Date=end

In [39]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [40]:
#calculated rf
#bond_yield = quandl.get('FRED/DGS3MO',api_key='X4EQzCpMfeaTCTDthRH_')
##bond_yield=np.log(bond_yield/100+1)
#date=list(map(str,bond_yield.index.values))
#date=list(map(modify_date,date))
#week=pd.to_datetime(date).strftime("%W").values
#week_1=np.roll(week,1)
#week_difference=np.where(week!=week_1)
#week_difference=np.delete(week_difference,0,None)
#rf_week=bond_yield.iloc[week_difference-1]

In [41]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)



In [42]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [43]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [44]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [45]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [46]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [47]:
# Read OPT-data
# optdata = pd.read_excel("optdata_SP500_2024_10_01.xlsx",index_col=0,sheet_name = 0)
optdata = pd.read_csv("optdata_SP500_20241202.csv",index_col=0)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index].astype(float)
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [48]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [49]:
preselected_stock

,Swan6,Swan7,Swan8,Swan9,Swan10D,Group,avg,PR,Rfwd_adjusted,std_f,SR,PR_SR
Ticker,,,,,,,,,,,,
AMGN,0.109292,0.843475,0.659977,0.526728,-0.417885,10,-1.208625,2.208625,0.055603,0.299317,0.185767,0.951933
AMT,0.050751,0.583639,-0.336659,0.882155,1.271482,3,1.312267,-0.312267,0.219806,0.260837,0.842696,-0.286656
BMY,-0.398965,1.296441,1.085742,0.931988,-0.566874,10,-1.841028,2.841028,0.055366,0.324001,0.170883,1.174424
DUK,-0.322793,0.233701,-0.356250,1.165454,0.534699,3,1.044984,-0.044984,0.210036,0.160198,1.311099,-0.051508
EXC,0.263389,0.605246,-0.123438,1.254518,0.353268,3,0.718945,0.281055,0.094612,0.203091,0.465859,0.191831
KO,0.325968,0.614677,-0.049526,1.161009,0.181359,3,0.523833,0.476167,0.152475,0.143303,1.064002,0.491168
LLY,0.280671,0.780342,0.061466,0.490481,-0.761718,10,-0.542674,1.542674,0.213233,0.303984,0.701462,1.292041
MCD,0.130072,0.483323,-0.202955,1.142392,0.376953,3,0.731855,0.268145,0.157775,0.206523,0.763958,0.234371
MDLZ,0.344205,0.089729,0.386481,0.928976,0.451681,3,0.756474,0.243526,0.162651,0.176498,0.921543,0.233778


In [50]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [51]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [52]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


OrderedDict([('AMGN', 0.0), ('AMT', 0.02675), ('BMY', 0.06587), ('DUK', 0.1), ('EXC', 0.0), ('KO', 0.09277), ('LLY', 0.1), ('MCD', 0.0607), ('MDLZ', 0.1), ('MRK', 0.07342), ('NEE', 0.0), ('PEP', 0.0), ('PG', 0.05274), ('SBUX', 0.02776), ('SO', 0.1), ('VZ', 0.1), ('WMT', 0.1)])
Expected annual return: 1.9%
Annual volatility: 10.0%
Sharpe Ratio: 0.09


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.019327318080995238, 0.09991913128491517, 0.09334867068048046)

In [53]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [54]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 0.09334867687964243


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [55]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.01]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [56]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  13 of 13 completed


In [57]:
stock_price.to_csv("tradeinfo.csv")
stock_price

Ticker,AMT,BMY,DUK,KO,LLY,MCD,MDLZ,MRK,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,
2024-12-02,207.330002,59.939999,115.180000,63.650002,799.799988,292.440002,65.230003,100.610001,179.699997,101.510002,87.769997,43.849998,92.639999
2024-12-03,207.789993,58.689999,113.639999,63.520000,813.330017,294.500000,64.809998,101.849998,175.419998,101.570000,86.260002,43.830002,93.510002
2024-12-04,207.500000,58.369999,113.279999,62.209999,829.840027,295.089996,63.270000,101.370003,175.110001,100.699997,85.849998,42.520000,94.449997


In [58]:
print(stock_price.head(1))
weight


Ticker             AMT        BMY     DUK         KO         LLY         MCD  \
Date                                                                           
2024-12-02  207.330002  59.939999  115.18  63.650002  799.799988  292.440002   

Ticker           MDLZ         MRK          PG        SBUX         SO  \
Date                                                                   
2024-12-02  65.230003  100.610001  179.699997  101.510002  87.769997   

Ticker             VZ        WMT  
Date                              
2024-12-02  43.849998  92.639999  


,weight
AMT,0.026750
BMY,0.065869
DUK,0.099999
KO,0.092769
LLY,0.099999
MCD,0.060699
MDLZ,0.099999
MRK,0.073419
PG,0.052739
SBUX,0.027760


In [59]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.17:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.17)
    row['weight']=0.17

for i,row in weight.iterrows():
  if row['weight']<0.165:
    row['weight']=row['weight']+row['weight']/rest*extra
    a=1

weight.to_csv("weights.csv")
weight

,weight
AMT,0.026750
BMY,0.065869
DUK,0.099999
KO,0.092769
LLY,0.099999
MCD,0.060699
MDLZ,0.099999
MRK,0.073419
PG,0.052739
SBUX,0.027760


In [60]:
weight = pd.read_csv("weights.csv",index_col=0)
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(1000000*weight.T)/np.array(stock_price.iloc[0,:]).T
weight

,weight
AMT,0.026750
BMY,0.065869
DUK,0.099999
KO,0.092769
LLY,0.099999
MCD,0.060699
MDLZ,0.099999
MRK,0.073419
PG,0.052739
SBUX,0.027760


In [61]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(1000000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 1000000/float(OEX[0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("Injected_result.csv")


<ipython-input-61-ff6f89a1631c>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 1000000/float(OEX[0])


In [62]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
#postallocationweight=post_allocation(dfMVOstock_sort)
#postallocationweight


In [63]:
#draw type E graph for postallocation


#